                                                # 01 - Exploration et nettoyage des données

                                                    **Projet DWFA, Drinking Water For All**

## **Contexte et objectif métier :**

Ce projet s'inscrit dans une mission de conseil pour DWFA (Drinking Water For All), une ONG d'accès à l'eau potable. L'objectif est d'aider un bailleur de fonds à décider où investir en priorité, en croisant les indicateurs disponibles sur 3 domaines d'expertise définis dans le compte rendu de réunion de lancement :

- 1. **Création de services** : développement de nouvelles infrastructures d'accès à l'eau potable dans les zones non couvertes.
- 2. **Modernisation des services** : amélioration de la qualité des services existants, notamment les infrastructures dites basiques, vers un niveau d'accès plus sécurisé.
- 3. **Consulting** : conseil aux gouvernements sur leurs politiques d'accès à l'eau, en tenant compte de la stabilité politique du pays.

Le tableau de bord final (Power BI, trois vues : mondiale, continentale, nationale) doit permettre d'identifier les pays prioritaires pour chacun de ces domaines. Cette exploration garde cet objectif en fil rouge : chaque fichier est examiné non seulement pour sa qualité technique, mais aussi pour ce qu'il apporte à l'un ou l'autre de ces trois domaines.

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_style("whitegrid")

## Inspection des cinq fichiers sources

Chaque fichier est inspecté selon la même méthode : chargement, inspection automatisée avec le module `inspection.py`, aperçu des premières lignes, puis traitement des alertes et réflexion sur les décisions à prendre en vue de l'élaboration du dashboard.

In [2]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "..", "src"))

from inspection import inspector

ModuleNotFoundError: No module named 'inspection'

---
## 1. Population, `Population.csv`  
---

In [ ]:
df_population = pd.read_csv("../DATA/Population.csv")

In [ ]:
inspector(df_population)


##########  INSPECTION : DataFrame  ##########

-------------------------------------------------------
STRUCTURE
-------------------------------------------------------
Lignes    : 20,914
Colonnes  : 5
Colonnes  : ['Key', 'Country', 'Year', 'Granularity', 'Population']

-------------------------------------------------------
TYPES DE DONNEES
-------------------------------------------------------
Key                str
Country            str
Year             int64
Granularity        str
Population     float64

-------------------------------------------------------
STATISTIQUES DESCRIPTIVES
-------------------------------------------------------
  Year
     min    : 2000
     max    : 2018
     mediane: 2009.0
     valeurs uniques : 19
  Population
     min    : 0.0
     max    : 1459377.612
     mediane: 3016.3365000000003
     valeurs uniques : 20515

-------------------------------------------------------
VALEURS MANQUANTES
-------------------------------------------------------
 

In [ ]:
df_population.head()

,Key,Country,Year,Granularity,Population
0,Afghanistan_2000,Afghanistan,2000,Total,20779.953
1,Afghanistan_2000,Afghanistan,2000,Male,10689.508
2,Afghanistan_2000,Afghanistan,2000,Female,10090.449
3,Afghanistan_2000,Afghanistan,2000,Rural,15657.474
4,Afghanistan_2000,Afghanistan,2000,Urban,4436.282


### **Résultat de l'inspection et conclusion**

L'inspection ne remonte aucune alerte : aucune valeur manquante, aucun doublon, aucune valeur hors plage physique. Les cinq granularités (Total, Male, Female, Rural, Urban) sont conservées telles quelles, sans filtrage sur Total uniquement, car le Domaine 1 (création de services) a besoin de croiser le taux d'accès à l'eau avec le taux de population urbaine pour orienter ses recommandations.

Ce fichier n'appelle donc aucune correction. Il servira aussi de base à la construction de la table pivot pays-année, présentée dans le notebook de préparation, car il couvre la période la plus large (2000 à 2018) et le plus grand nombre de pays.

---
## **2. Services d'eau,** `BasicAndSafelyManagedDrinkingWaterServices.csv`  
---

In [ ]:
df_water_services = pd.read_csv("../DATA/BasicAndSafelyManagedDrinkingWaterServices.csv")

In [ ]:
inspector(df_water_services)


##########  INSPECTION : DataFrame  ##########

-------------------------------------------------------
STRUCTURE
-------------------------------------------------------
Lignes    : 10,476
Colonnes  : 5
Colonnes  : ['Year', 'Country', 'Granularity', 'Population using at least basic drinking-water services (%)', 'Population using safely managed drinking-water services (%)']

-------------------------------------------------------
TYPES DE DONNEES
-------------------------------------------------------
Year                                                             int64
Country                                                            str
Granularity                                                        str
Population using at least basic drinking-water services (%)    float64
Population using safely managed drinking-water services (%)    float64

-------------------------------------------------------
STATISTIQUES DESCRIPTIVES
-------------------------------------------------------

In [ ]:
df_water_services.head()

,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.61913,NaN
1,2000,Afghanistan,Total,27.77190,NaN
2,2000,Afghanistan,Urban,49.48745,NaN
3,2000,Albania,Rural,81.78472,NaN
4,2000,Albania,Total,87.86662,49.29324


### **Définitions des indicateurs :**

- "At least basic" désigne l'accès à une source d'eau améliorée (puits protégé, source aménagée, eau en bouteille), à condition que le temps de collecte aller-retour, attente comprise, ne dépasse pas 30 minutes. C'est le critère minimum retenu par l'OMS (Organisation Mondiale de la Santé).  
- "Safely managed" désigne l'accès à une source améliorée située sur place, disponible lorsque nécessaire et exempte de contamination, un critère nettement plus exigeant.  
  
Cette différence de niveau d'exigence explique pourquoi les pourcentages de `safely_managed_water` comportent davantage de valeurs manquantes que les pourcentages de `basic_water` : la source ne renseigne pas cet indicateur pour de nombreux pays. Rien n'indique une erreur de saisie : la valeur est absente, et le jeu de données n'en précise pas la raison.


### **Valeurs supérieures à 100 % :**

7 lignes affichent 100,00001 % au lieu de 100,000 % pour l'Islande, l'Andorre, la Finlande, les Palaos et la Norvège. Il s'agit d'une erreur d'arrondi en amont, en virgule flottante, et non d'une erreur de saisie : ces pays ont réellement un accès universel à l'eau.

**Décision : plafonnement à 100 (`.clip`) plutôt que suppression.** Une valeur supérieure à 100 % est physiquement impossible pour un pourcentage, et se remarquerait immédiatement sur un graphique dont l'axe va jusqu'à 100. Supprimer la ligne ferait perdre un pays valide pour une erreur mineure d'arrondi, le plafonnement est donc préférable.  


In [ ]:
col_basic = "Population using at least basic drinking-water services (%)"
col_safe = "Population using safely managed drinking-water services (%)"

df_water_services[col_basic] = df_water_services[col_basic].clip(upper=100)
df_water_services[col_safe] = df_water_services[col_safe].clip(upper=100)

assert df_water_services[col_basic].max() <= 100
assert df_water_services[col_safe].max() <= 100
print("Correction appliquée, plus de valeur supérieure à 100")

Correction appliquée, plus de valeur supérieure à 100


### **Précision décimale :**

Les deux colonnes de pourcentage contiennent jusqu'à 5 décimales (par exemple 27,77190). Le projet demande de classer les pays par domaine d'expertise, un exercice de comparaison directe où 2 pays proches (79,44 % et 79,68 %) peuvent changer d'ordre selon le niveau d'arrondi retenu.

Arrondir trop tôt dans le pipeline Python présente un risque : si l'écart réel entre 2 pays se joue à la troisième ou quatrième décimale, un arrondi précoce à 2 décimales peut faire basculer un classement sans justification analytique, uniquement parce que la précision a été perdue avant même le calcul.

**Décision : la précision native de la donnée est conservée telle quelle dans les fichiers exportés.** Aucun arrondi n'est appliqué en Python. L'affichage à 2 décimales, utile pour la lisibilité du dashboard, sera géré dans Power BI au niveau du format des mesures, une couche qui ne modifie pas la donnée sous-jacente utilisée pour les calculs et les classements. Cette même règle s'applique à tous les indicateurs numériques du projet (mortalité, stabilité politique).

### **Conclusion :**

Ce fichier alimentera directement les indicateurs des trois domaines d'expertise, en particulier les nuages de points croisant accès à l'eau et population urbaine (Domaine 1), ou niveau basique et niveau sécurisé (Domaine 2). La conservation des granularités Rural, Urbain et Total permettra ces filtres directement dans le dashboard.

---
## 3. **Mortalité liée à l'eau,** `MortalityRateAttributedToWater.csv`

Source : Organisation mondiale de la santé (OMS).  
  
  
---

In [ ]:
df_mortality = pd.read_csv("../DATA/MortalityRateAttributedToWater.csv")

In [ ]:
inspector(df_mortality)


##########  INSPECTION : DataFrame  ##########

-------------------------------------------------------
STRUCTURE
-------------------------------------------------------
Lignes    : 549
Colonnes  : 5
Colonnes  : ['Year', 'Country', 'Granularity', 'Mortality rate attributed to exposure to unsafe WASH services', 'WASH deaths']

-------------------------------------------------------
TYPES DE DONNEES
-------------------------------------------------------
Year                                                               int64
Country                                                              str
Granularity                                                          str
Mortality rate attributed to exposure to unsafe WASH services    float64
WASH deaths                                                      float64

-------------------------------------------------------
STATISTIQUES DESCRIPTIVES
-------------------------------------------------------
  Year
     min    : 2016
     max    

In [ ]:
df_mortality.head()

,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31193,NaN
1,2016,Afghanistan,Male,12.61297,NaN
2,2016,Afghanistan,Total,13.92067,4824.353
3,2016,Albania,Female,0.12552,NaN
4,2016,Albania,Male,0.20650,NaN


### **Précisions indicateurs :**

2 indicateurs coexistent dans ce fichier, avec des unités différentes :  
  
- Le taux de mortalité est exprimé pour 100 000 habitants (une valeur de 13,92 signifie 13,92 décès pour 100 000 habitants).  
- Le nombre de décès WASH est une estimation statistique (une valeur de 4824,353 correspond à un nombre de décès attribuables estimé par l'OMS, pas à un comptage direct).

Ce taux n'est pas un pourcentage, contrairement aux indicateurs d'accès à l'eau : sa borne supérieure logique n'est donc pas 100. Un pays très affecté peut légitimement la dépasser, le fichier source contient d'ailleurs un maximum réel de 107,05.

### **Décision : aucun plafonnement.**   
Le Domaine 3 (consulting) combine mortalité et accès à l'eau pour construire un indicateur d'efficacité gouvernementale. Plafonner ce taux à 100 par réflexe, comme pour un pourcentage, écraserait artificiellement les pays les plus touchés au même niveau que des pays bien moins affectés, ce qui fausserait exactement l'indicateur destiné à orienter la décision d'investissement.

### **`WASH deaths` : ni arrondi ni conversion en entier :**

`WASH deaths` n'est pas un comptage direct de décès individuels, mais une **fraction attribuable** calculée par l'OMS : le nombre total de décès dus à une cause proche (maladies diarrhéiques, par exemple), multiplié par la proportion estimée de ces décès imputable à un accès à l'eau non sécurisé. Ce calcul produit un nombre décimal par construction, et non un artefact d'arrondi flottant comme sur les pourcentages d'accès à l'eau.

**Décision : la précision native de cette colonne est conservée**, sans arrondi ni conversion en type entier. Sur 183 valeurs renseignées, 14 sont inférieures à 0,5 et disparaîtraient complètement (arrondi à 0) si un arrondi était appliqué en Python, effaçant l'information pour des pays dont l'effectif estimé est réellement faible mais non nul. L'affichage à un nombre entier, si souhaité pour la lisibilité, sera géré au niveau du format de la mesure dans Power BI, une couche qui ne modifie pas la donnée sous-jacente.

  
- Les valeurs manquantes, sur le taux comme sur le nombre de décès, ne sont pas renseignées par la source, qui n'en précise pas la raison.  
  
**Décision :** conservation des valeurs manquantes, sans imputation : pour la même raison que sur les indicateurs d'eau : inventer un chiffre de mortalité serait le biais le plus grave possible sur ce projet.

### **Limite de couverture temporelle :**

**Ce fichier ne couvre que l'année 2016**, contrairement aux autres sources qui vont de 2000 à 2018. Toute analyse combinant la mortalité avec un autre indicateur sera donc une photographie ponctuelle de 2016, et non une évolution temporelle. Cette limite sera présentée explicitement dans le PowerPoint comme une preuve de rigueur analytique plutôt que dissimulée.

In [ ]:
col_rate = "Mortality rate attributed to exposure to unsafe WASH services"

print("Max du taux de mortalité, non plafonné :", df_mortality[col_rate].max())
df_mortality.head(10)

Max du taux de mortalité, non plafonné : 107.04802


,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31193,NaN
1,2016,Afghanistan,Male,12.61297,NaN
2,2016,Afghanistan,Total,13.92067,4824.35300
3,2016,Albania,Female,0.12552,NaN
4,2016,Albania,Male,0.20650,NaN
5,2016,Albania,Total,0.16641,4.86975
6,2016,Algeria,Female,2.19890,NaN
7,2016,Algeria,Male,1.72837,NaN
8,2016,Algeria,Total,1.86723,758.21000
9,2016,Angola,Female,45.15024,NaN


### **Conclusion :**

Cet indicateur alimente le Domaine 3, en particulier les nuages de points croisant efficacité politique et stabilité. La limite d'année unique (2016) sera rappelée à chaque utilisation de cet indicateur dans le dashboard.

---
## 4. **Stabilité politique :** `PoliticalStability.csv`  
---

In [ ]:
df_political_stability = pd.read_csv("../DATA/PoliticalStability.csv")

In [ ]:
inspector(df_political_stability)


##########  INSPECTION : DataFrame  ##########

-------------------------------------------------------
STRUCTURE
-------------------------------------------------------
Lignes    : 3,526
Colonnes  : 4
Colonnes  : ['Country', 'Year', 'Political_Stability', 'Granularity']

-------------------------------------------------------
TYPES DE DONNEES
-------------------------------------------------------
Country                    str
Year                     int64
Political_Stability    float64
Granularity                str

-------------------------------------------------------
STATISTIQUES DESCRIPTIVES
-------------------------------------------------------
  Year
     min    : 2000
     max    : 2018
     mediane: 2010.0
     valeurs uniques : 18
  Political_Stability
     min    : -3.31
     max    : 1.97
     mediane: 0.05
     valeurs uniques : 444

-------------------------------------------------------
VALEURS MANQUANTES
-------------------------------------------------------
  A

In [ ]:
df_political_stability.head()

,Country,Year,Political_Stability,Granularity
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total
3,Afghanistan,2004,-2.30,Total
4,Afghanistan,2005,-2.07,Total


### **Traitement de l'alerte sur les valeurs négatives :**

L'inspection automatisée signale 1673 valeurs négatives comme physiquement impossibles. Cette alerte est un faux positif : la règle générique du module suppose qu'une valeur négative est une erreur, ce qui est vrai pour un pourcentage ou un effectif de population, mais faux ici. `Political_Stability` est un indice de gouvernance de type Banque mondiale (Worldwide Governance Indicators), centré sur 0 et s'étendant approximativement **de -3,3 à +2,0 : une valeur négative signale une instabilité politique, une valeur positive une stabilité.** C'est le sens même de l'indicateur, pas une anomalie de saisie.

**Décision :** aucune correction sur les valeurs négatives. Le Domaine 3 (consulting) utilise justement cet indice pour écarter les pays les plus instables. Corriger ces valeurs, par exemple en les plafonnant à 0, supprimerait l'information qui permet précisément de repérer l'instabilité, à l'inverse de l'objectif recherché.

In [ ]:
print("Min :", df_political_stability["Political_Stability"].min())
print("Max :", df_political_stability["Political_Stability"].max())
print(
    "Part de valeurs négatives :",
    (df_political_stability["Political_Stability"] < 0).mean().round(3),
    "(attendu, pas une anomalie)",
)

Min : -3.31
Max : 1.97
Part de valeurs négatives : 0.474 (attendu, pas une anomalie)


### **Les valeurs hors de la fenêtre −2,5 / +2,5 :**

La Banque mondiale publie cet indice sur une échelle standard normale, « *approximately –2.5 to 2.5* » (*WGI 2025 Methodology Revision*, p. 43). L'inspection ci-dessus affiche pourtant un minimum à −3,31. Avant d'aller plus loin, il faut savoir combien de valeurs sortent de cette fenêtre, lesquelles, et ce qu'elles signifient.


In [ ]:
fenetre = 2.5
hors_fenetre = df_political_stability[
    df_political_stability["Political_Stability"].abs() > fenetre
]

print(
    f"Valeurs hors de la fenêtre ±{fenetre} : {len(hors_fenetre)} "
    f"sur {len(df_political_stability)} ({len(hors_fenetre) / len(df_political_stability):.2%})"
)
print("Valeurs au-dessus de +2,5 :", (hors_fenetre["Political_Stability"] > 0).sum())
print()

recap = (
    hors_fenetre.groupby("Country")
    .agg(
        minimum=("Political_Stability", "min"),
        premiere_annee=("Year", "min"),
        derniere_annee=("Year", "max"),
        nb_annees=("Year", "size"),
    )
    .sort_values("minimum")
)
print(recap.to_string())


Valeurs hors de la fenêtre ±2.5 : 49 sur 3526 (1.39%)


Valeurs au-dessus de +2,5 : 0

                          minimum  premiere_annee  derniere_annee  nb_annees
Country                                                                     
Somalia                     -3.31            2003            2014         12
Iraq                        -3.18            2004            2018          5
Yemen                       -3.00            2014            2018          5
Syrian Arab Republic        -2.97            2012            2018          7
Pakistan                    -2.81            2008            2013          6
Afghanistan                 -2.80            2008            2018          8
Central African Republic    -2.70            2014            2014          1
Sudan                       -2.67            2009            2011          3
South Sudan                 -2.65            2014            2014          1
Burundi                     -2.52            2004            2004          1


### **Ce que sont ces valeurs, et ce qu'on en fait :**

Les 49 valeurs concernées, soit 1,4 % du fichier, sont **toutes du côté négatif** et se concentrent sur dix pays : Somalie, Irak, Yémen, Syrie, Pakistan, Afghanistan, République centrafricaine, Soudan, Soudan du Sud et Burundi.

Trois éléments expliquent leur existence, et aucun n'est une anomalie de saisie :

1. **La fenêtre ±2,5 est indicative, pas une borne.** Le document de référence écrit « approximately ». Rien dans le modèle ne tronque l'échelle standard normale ; la seule troncature qu'il mentionne, p. 38, porte sur les intervalles de confiance du score absolu 0-100, qui n'est pas l'échelle utilisée ici.
2. **±2,5 correspond à 2,5 écarts-types.** Sur une loi normale centrée réduite, 1,24 % des valeurs se situent hors de cet intervalle ; le fichier en compte 1,39 %. C'est la queue de distribution attendue, pas un défaut.
3. **L'indicateur mesure explicitement l'absence de violence** : la dimension s'appelait auparavant *Political Stability and Absence of Violence/Terrorism* (*ibid.*, p. 4). Une valeur très négative décrit donc un conflit armé ouvert, ce qui correspond exactement aux pays et aux périodes que le tableau ci-dessus fait ressortir.

**Décision : les valeurs sont conservées telles quelles, sans plafonnement.**

Le plafonnement appliqué aux pourcentages d'accès à l'eau ne constitue pas un précédent. Un pourcentage de population **ne peut pas** dépasser 100 : les valeurs à 100,0000001 étaient des artefacts de calcul flottant, et les ramener à 100 rétablissait une vérité arithmétique. Ici, −3,31 est une valeur publiée, exacte, et porteuse de sens : la ramener à −2,5 remplacerait une donnée réelle par une donnée inventée, ce que ce projet s'interdit partout ailleurs, à commencer par les valeurs manquantes.

Deux conséquences concrètes achèvent de trancher :

- **un plafonnement casserait le filtre du tableau de bord.** Le compte rendu de réunion demande un filtre permettant d'écarter les pays trop instables. Si la Somalie, l'Irak et la Syrie étaient toutes ramenées à −2,5, elles deviendraient indiscernables des pays simplement très instables, et un utilisateur plaçant le curseur juste au-dessus de −2,5 les conserverait toutes ;
- **l'écart entre −2,5 et −3,31 est une information de priorisation.** Entre un pays très instable et un État effondré, la nature de l'intervention diffère : le premier relève encore du consulting auprès d'une administration, le second relève de l'urgence humanitaire.

Deux précisions de lecture, pour que les chiffres se recoupent :

- le maximum affiché ci-dessus, +1,97, porte sur le fichier brut. Après restriction aux 194 États membres de l'OMS, au notebook 02, il devient +1,76 : les territoires écartés comptaient parmi les plus stables ;
- dans le dashboard, le curseur de stabilité politique s'étend donc de −3,31 à +1,76, et non de −2,5 à +2,5. L'écart est documenté dans la page Documentation du rapport.


### **Conclusion :**

Ce fichier ne comporte qu'une seule granularité (Total), aucune valeur manquante et aucun doublon. Aucune correction n'est nécessaire au-delà de la vérification ci-dessus. Il alimentera le filtre de stabilité politique du dashboard, ainsi que le Domaine 3.

---
## 5. **Correspondance pays et région :** `RegionCountry.csv`  
---

Ce fichier associe chaque pays à l'une des six régions officielles de découpage administratif de l'OMS, elles ne correspondent pas à des continents.  
La région Eastern Mediterranean, par exemple, regroupe des pays comme l'Afghanistan, le Pakistan ou la Somalie, bien au-delà du seul pourtour méditerranéen. De même, les Amériques du Nord, du Sud et les Caraïbes forment une seule région OMS, alors qu'un découpage par continent les séparerait.

**Décision :** Conserver ce découpage régional officiel garantit la cohérence avec la nomenclature déjà utilisée dans le fichier de mortalité, sans nécessiter de mapping supplémentaire entre continents et régions OMS.

In [ ]:
df_region_country = pd.read_csv("../DATA/RegionCountry.csv")

In [ ]:
inspector(df_region_country)


##########  INSPECTION : DataFrame  ##########

-------------------------------------------------------
STRUCTURE
-------------------------------------------------------
Lignes    : 194
Colonnes  : 2
Colonnes  : ['REGION (DISPLAY)', 'COUNTRY (DISPLAY)']

-------------------------------------------------------
TYPES DE DONNEES
-------------------------------------------------------
REGION (DISPLAY)     str
COUNTRY (DISPLAY)    str

-------------------------------------------------------
STATISTIQUES DESCRIPTIVES
-------------------------------------------------------
  Aucune colonne numerique detectee.

-------------------------------------------------------
VALEURS MANQUANTES
-------------------------------------------------------
  Aucun NaN dans ce fichier.

  Aucune colonne entierement vide.
  Aucune chaine vide ('').

-------------------------------------------------------
DOUBLONS
-------------------------------------------------------
  Aucun doublon.

-------------------------

In [ ]:
df_region_country.head()

,REGION (DISPLAY),COUNTRY (DISPLAY)
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia
3,Western Pacific,Australia
4,Europe,Austria


### **Conclusion :**

Ce fichier fait office de table de correspondance géographique fixe, sans dimension temporelle. Il servira à agréger les indicateurs par région dans la vue continentale du dashboard.

---
## **Pistes d'enrichissement envisagée et non retenue :**  
---

### **- 'Infrastructures aéroportuaires** *Datagouv (Upply Airports)*

L'ajout des données d'infrastructures aéroportuaires (Datagouv, Upply Airports) a été envisagé. Cette source décrit la densité et la qualité des équipements aéroportuaires par pays, et aurait pu servir d'indicateur indirect de la capacité logistique d'un pays à recevoir et déployer du matériel dans le cadre d'un projet d'accès à l'eau.

**Décision : non retenu**, pour les raisons suivantes :

- le lien entre infrastructures aéroportuaires et accès à l'eau potable n'est pas démontré par les données disponibles ici, l'hypothèse est plausible mais non mesurable avec ce jeu de données.
- les 3 domaines d'expertise (création de services, modernisation, consulting) sont déjà correctement couverts par les indicateurs eau, mortalité, stabilité politique et population, sans qu'un axe logistique supplémentaire soit nécessaire pour y répondre.
- le transport aérien est surtout un indicateur de connectivité internationale, alors qu'un projet de déploiement d'infrastructures d'eau relève davantage d'une capacité d'intervention locale : un indicateur couvrant plus largement les routes, ports et douanes serait a priori plus pertinent pour ce type de problématique.

### **- 'Indice de performance logistique (LPI)'** *Banque mondiale (data.worldbank.org)*

Cette piste plus large, couvrant précisément les routes, ports et douanes évoqués ci-dessus, a donc également été envisagée, via le LPI (Logistics Performance Index) disponible sur le World Bank DataBank (indicateur LP.LPI.OVRL.XQ). Cet indicateur mesure la qualité des infrastructures logistiques d'un pays et aurait pu constituer un facteur explicatif supplémentaire de la difficulté à déployer des projets d'accès à l'eau.

**Décision : non retenu**, pour les raisons suivantes :

- le lien entre performance logistique et accès à l'eau potable n'est pas démontré par les données disponibles ici, l'hypothèse resterait tout aussi spéculative que celle des infrastructures aéroportuaires, l'ajouter reviendrait à insérer une corrélation supposée plutôt qu'un indicateur validé.
- les 3 domaines d'expertise restent déjà correctement couverts par les indicateurs eau, mortalité, stabilité politique et population.
- ajouter cette source aurait complexifié le modèle de données sans gain analytique réel, et aurait dilué la lisibilité du dashboard pour le bailleur de fonds.

**Le périmètre retenu reste donc celui des cinq jeux de données fournis, en privilégiant la profondeur d'analyse plutôt que la multiplication de sources externes non directement liées à la problématique.**

---
## **Export des fichiers nettoyés**  
---

Chaque fichier est exporté tel quel après correction des erreurs identifiées ci-dessus, sans arrondi ni restructuration. La construction de la clé de jointure, de la table pivot pays-année et la sélection finale des colonnes sont réalisées dans le notebook `02_preparation_datasets.ipynb`, dédié à la préparation du modèle pour Power BI.

In [ ]:
exports = {
    "population_clean.csv": df_population,
    "water_services_clean.csv": df_water_services,
    "mortality_clean.csv": df_mortality,
    "political_stability_clean.csv": df_political_stability,
    "region_country_clean.csv": df_region_country,
}

for filename, df in exports.items():
    path = f"../DATA/{filename}"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{filename}, {df.shape[0]} lignes, {df.shape[1]} colonnes")

population_clean.csv, 20914 lignes, 5 colonnes


water_services_clean.csv, 10476 lignes, 5 colonnes
mortality_clean.csv, 549 lignes, 5 colonnes
political_stability_clean.csv, 3526 lignes, 4 colonnes
region_country_clean.csv, 194 lignes, 2 colonnes
